# DeepfakeDetector — Colab Training

Before running: in Colab go to **Runtime → Change runtime type → T4 GPU** (or any available GPU).

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Hanineyasser/DeepfakeDetector.git"
REPO_DIR = Path("/content/DeepfakeDetector")

if not Path("convnext_trainer.py").exists():
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
else:
    !git pull

print("Working directory:", os.getcwd())

In [ ]:
from pathlib import Path
import shutil

from google.colab import drive
drive.mount("/content/drive")

for local_name, drive_name in [
    ("models", "DeepfakeDetector_models"),
    ("lightning_logs", "DeepfakeDetector_logs"),
]:
    drive_dir = Path("/content/drive/MyDrive") / drive_name
    drive_dir.mkdir(parents=True, exist_ok=True)

    local_dir = Path(local_name)
    if local_dir.is_symlink():
        local_dir.unlink()
    elif local_dir.exists():
        shutil.rmtree(local_dir)

    local_dir.symlink_to(drive_dir)
    print(f"{local_name}/ -> {drive_dir}")

In [ ]:
!pip install -q pytorch-lightning kagglehub

In [ ]:
import os
from pathlib import Path
import kagglehub
import yaml

dataset_path = Path(kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces"))
base = dataset_path / "real_vs_fake" / "real-vs-fake"

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["train_paths"] = [str(base / "train")]
cfg["val_paths"] = [str(base / "valid")]
cfg["batch_size"] = 64
cfg["num_workers"] = min(4, os.cpu_count() or 2)

last_ckpt = Path("models/last.ckpt")
cfg["resume_from_checkpoint"] = str(last_ckpt) if last_ckpt.exists() else None

with open("config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Train path:", cfg["train_paths"])
print("Val path:", cfg["val_paths"])
print("Batch size:", cfg["batch_size"], "| num_workers:", cfg["num_workers"])
print("Resume from checkpoint:", cfg["resume_from_checkpoint"])

In [ ]:
import torch
from convnext_trainer import main

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

main()